# EXP-2026-003: diagnose MSF conditioning on frozen weights

This notebook calls Python modules; it does not implement a model or training loop.
Read `experiments/registry/EXP-2026-003.md` for the approved plan.

Zero training steps. The order is fixed:

1. **D0, parent replay:** the author checkpoint and Stage5 r2 repeat the parent's own validation procedure on identical inputs and noise, and the result is compared with the parent's record. If it does not reproduce, the reading below says so and makes no claim about the model.
2. **D1, paired usefulness:** the same target, noise and text with own, donor or zero pre-op channels, reported per timestep, patient and ROI. A donor is the next patient's pre-op slice at the same physical level.
3. **D2, generation:** the factorial of channels and starting pair, the pure-noise start and the guidance sweep. Conclusions about free-running anatomy or guidance need this phase, so `PHASE = "full"` is the default. `"loss"` is only a debugging option.
4. **Reading:** the decision table frozen in the card, including mixed and inconclusive outcomes. It is never a quality pass and does not by itself establish a learning-rate or memorization cause.

- All text is **target-informed class text (oracle)**. Added residual-positive validation sentinels are explicitly oracle-selected scoring cases, separate from the frozen previews.
- Original MSF is a text-only control, not a patient-preserving baseline by design. Stage5 Part2 stays pending; this diagnostic does not replace it.
- Inputs are read from approved Drive storage and staged locally. Outputs are written locally, uploaded by Drive folder ID after each stage and confirmed by server MD5 at the end.
- Runtime has not been measured. Estimate: 30 to 60 minutes.
- Keep all outputs private. **Clear notebook outputs before committing/sharing.** No token should be pasted into this notebook or chat.


In [ ]:
REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/MedSegFactory.git"
BRANCH = "exp-2026-003-msf-diagnostics"
WORK = "/content/MedSegFactory-msf-diagnostics"
RUN_TAG = "diagnostic_r1"  # immutable; change after any attempt that created output
PHASE = "full"  # the plan; "loss" alone is a debugging option
DRIVE_RHUH = "/content/drive/MyDrive/RHUH-GBM"
DATA_ON_DRIVE = ("RHUH-GBM", "synthesis", "EXP-2026-001")
PARENT_RUN = ("stage5_runs", "r2")

## 1. Clone and install the diagnostic dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if not Path(WORK).exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK], check=True)
else:
    current = subprocess.check_output(["git", "-C", WORK, "branch", "--show-current"], text=True).strip()
    assert current == BRANCH, "This checkout belongs to another task; choose a different WORK"
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
    "SimpleITK==2.5.6", "nibabel==5.4.2", "pydicom==3.0.2",
    "numpy==2.4.6", "scipy==1.16.3", "matplotlib>=3.9,<4", "pillow"], check=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

## 2. Read parent artifacts by Drive ID and verify local output storage
The preflight checks output access before model loading. Existing runs are never reused.

In [ ]:
import json
import shutil
from google.colab import auth, drive
from googleapiclient.discovery import build
from rhuh_synthesis.drive_api import DriveRun
from rhuh_synthesis.manifest import file_sha256

assert PHASE in ("loss", "full")
assert RUN_TAG and RUN_TAG not in (".", "..") and Path(RUN_TAG).name == RUN_TAG
config = json.loads(Path("configs/experiments/EXP-2026-003.json").read_text())
LOCAL_BASE = Path("/content/msf003_data")
DATASET = LOCAL_BASE / "EXP-2026-001"
OUTPUT = LOCAL_BASE / "EXP-2026-003" / RUN_TAG
PARENT = Path("/content/msf003_parent_r2")
assert not OUTPUT.exists(), "Choose a new RUN_TAG; previous attempts remain immutable"

drive.mount("/content/drive")
auth.authenticate_user()
api = build("drive", "v3", cache_discovery=False)
parent_drive = DriveRun.open(api, DATA_ON_DRIVE, PARENT_RUN, PARENT, create=False)
for name in ("record.json", "record.json.sha256", "train_log.jsonl"):
    parent_drive.fetch(name)
assert file_sha256(PARENT / "record.json") == config["parent_record_sha256"], "Unexpected parent record"
parent = json.loads((PARENT / "record.json").read_text())
TRAINED_WEIGHTS = parent_drive.fetch(parent["weights"]["file"])
assert file_sha256(TRAINED_WEIGHTS) == config["adapted_weights_sha256"], "Unexpected adapted weights"
parent_drive.fetch_folder("previews")

run_drive = DriveRun.open(api, ("RHUH-GBM", "synthesis"), ("EXP-2026-003", RUN_TAG), OUTPUT,
                          expect_in_base=("EXP-2026-001",), create=True)
preflight_drive = DriveRun(api, run_drive.folder_id, LOCAL_BASE / "preflight" / RUN_TAG)
preflight = preflight_drive.preflight()
print("Drive write/read/MD5 preflight passed; free GiB:", preflight["free_gib"])

## 3. Stage only the selected training/validation exports locally
The parent hash is verified before reading its selection. No test exports are copied or loaded.

In [ ]:
PRIVATE = Path(DRIVE_RHUH) / "synthesis" / "EXP-2026-001"
split = json.loads((PRIVATE / "split/split.json").read_text())
selections = {"train": parent["selection"], "val": parent["val_selection"]}
for group, entries in selections.items():
    allowed = set(split["roles"][group]["patients"])
    assert {e["patient"] for e in entries} <= allowed, "Selection crosses split"
    assert not {e["patient"] for e in entries} & set(split["roles"]["test"]["patients"])
    for entry in entries:
        patient = entry["patient"]
        assert Path(patient).name == patient and patient not in (".", "..")
        options = list(PRIVATE.glob(f"export_v1/train*/{patient}.npz")) if group == "train" else [PRIVATE / "export_v1/val" / f"{patient}.npz"]
        assert len(options) == 1 and options[0].is_file(), "Missing or ambiguous export"
        for source in (options[0], options[0].with_suffix(".json")):
            target = DATASET / source.relative_to(PRIVATE)
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists() or file_sha256(target) != file_sha256(source):
                shutil.copyfile(source, target)
for relative in ("split/split.json", "registration_policy/acceptance_policy_v1.json"):
    target = DATASET / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(PRIVATE / relative, target)
print("Selected exports staged locally; test set untouched")

## 4. Audit the parent and fetch the original MSF checkpoint
No VLM access or MedGemma token is required. The author checkpoint must match the frozen hash.

In [ ]:
from huggingface_hub import hf_hub_download
from rhuh_synthesis.msf_diagnostics import audit_parent_run, run_msf_diagnostic

parent_audit = audit_parent_run(PARENT)
print(json.dumps(parent_audit, indent=2))
spec = config["original_checkpoint"]
ORIGINAL_CHECKPOINT = Path(hf_hub_download(spec["model_id"], spec["filename"]))
assert file_sha256(ORIGINAL_CHECKPOINT) == spec["sha256"], "Unexpected original MSF checkpoint"

## 5. Run the frozen diagnostic
Replay, D1 and D2 run in that order, with an upload after each stage. A failed attempt keeps its files and needs a new run tag.


In [ ]:
def upload(stage):
    report = run_drive.sync()
    print(f"Drive upload after {stage}: {report['uploaded']} new, {report['kept']} kept, {len(report['failed'])} failed")
    for name, reason in report["failed"].items():
        print("  will retry:", name, reason)


record = run_msf_diagnostic(
    "configs/experiments/EXP-2026-003.json",
    dataset_root=DATASET, parent_run_dir=PARENT,
    trained_weights=TRAINED_WEIGHTS, original_checkpoint=ORIGINAL_CHECKPOINT,
    output_dir=OUTPUT, phase=PHASE, device="cuda", on_checkpoint=upload,
)
print(json.dumps({k: record.get(k) for k in
                  ("experiment_id", "status", "phase", "training_steps", "seconds", "peak_allocated_gib",
                   "loss_row_count", "sampling_attempted", "sampling_pairs", "replay_status")}, indent=2))


## 6. Read the result

**Replay first.** If `replay_status` is not `reproduced`, the decision row says to resolve the instrumentation, and nothing below supports a claim about the model.

**D1:** the ratio is shuffled error over own error at the same target and noise, shown with its direction. `+` means corruption raised the error in both noise draws, beyond numerical repeatability and the draw-to-draw spread. Empty ROIs stay `NA`; these are teacher-forced numbers, not free-generation success rates.

**D2:** raw metrics per arm and patient, one draw per case. **Decision:** the row of the frozen table. These are diagnostics on three validation patients, not clinical results.


In [ ]:
from rhuh_synthesis.msf_diagnostic_probe import format_d1_table, format_d2_table
from rhuh_synthesis.msf_diagnostics import shareable_readout

print("REPLAY:", record["replay_status"])
for model, entry in record["replay"]["models"].items():
    print(" ", model, "fixed loss max relative difference:", entry["fixed_loss"]["max_relative_difference"],
          "| repeatability:", entry["repeatability"]["max_relative_difference"],
          "| generation:", entry.get("generation"))
print(" inputs identical:", record["replay"]["inputs_identical"], "| saved previews rescored:", record["replay"]["saved_preview_rescore"])
print()
print(format_d1_table(record["d1_reading"]))
print()
if record["sampling_summary"]:
    print(format_d2_table(record["sampling_summary"]))
    print()
    print("D2 cell:", record["d2_reading"]["cell"], "| guidance:", {arm: value["reading"] for arm, value in record["d2_reading"]["guidance"].items()})
print()
print("DECISION:", json.dumps(record["decision"], indent=1))
print()
from rhuh_synthesis.manifest import write_once_json

share_path, _ = write_once_json(OUTPUT / "shareable.json", shareable_readout(record))
print("The part to send is saved as", share_path.name, "(patient letters only, no patient IDs)")


## 7. Confirm persistence
Do not close this session until every result file is confirmed by the Drive server. No automatic training follows this diagnostic.

Then download `shareable.json`, `record.json` and the `grids` folder from `RHUH-GBM/synthesis/EXP-2026-003/<RUN_TAG>/` on Drive into `data/RHUH-GBM/synthesis/EXP-2026-003/<RUN_TAG>/` on your laptop. The `arrays` folder can stay on Drive. If the session stops part way, keep whatever was uploaded and start again with a new `RUN_TAG`.


In [ ]:
assert run_drive.confirm(), f"Some uploads are unconfirmed; keep this session and local files at {OUTPUT}"
print("All diagnostic files confirmed by server MD5")